# Tutorial: Who Killed Daphine?

This tutorial introduces the core ProvSQL features through a self-contained crime mystery.

## The Scenario

A group of 20 persons spent the night in a manor. In the morning, Daphine, a young lady, went missing. Her dead body was found the following day in the cellar. The autopsy revealed the following facts:

- She died of a head injury caused by a blunt-force instrument.
- Her death happened between midnight and 8 am.
- She was not killed in the cellar, but her body was moved there afterwards.
- Goose down found in her wound proves that she died in bed in one of the bedrooms; unfortunately all beds have the same down pillows, and pillowcases were changed in the morning, so it is impossible to identify which bedroom.

The police interviewed all 19 suspects and collected their statements about who they saw in which room at which time. They also assessed each witness’s reliability through a psychological evaluation.

**Your mission:** help the police discover who killed Daphine, using the power of provenance management and probabilistic databases.

## Setup

*The following cells set up the database with all the content this notebook requires; run them first, ideally on a fresh database.*

Example tables

In [ ]:
DROP TABLE IF EXISTS public.person CASCADE;
CREATE TABLE public.person (
    id integer NOT NULL,
    name text NOT NULL,
    date_of_birth date,
    height smallint
);

In [ ]:
DROP TABLE IF EXISTS public.reliability CASCADE;
CREATE TABLE public.reliability (
    person integer NOT NULL,
    score double precision NOT NULL
);

In [ ]:
DROP TABLE IF EXISTS public.room CASCADE;
CREATE TABLE public.room (
    id integer NOT NULL,
    name text NOT NULL,
    area smallint
);

In [ ]:
DROP TABLE IF EXISTS public.sightings CASCADE;
CREATE TABLE public.sightings (
    "time" time without time zone NOT NULL,
    person integer NOT NULL,
    room integer NOT NULL,
    witness integer
);

In [ ]:
TRUNCATE public.person;
COPY public.person (id, name, date_of_birth, height) FROM stdin;
0	Titus	1969-04-03	163
1	Norah	1983-10-15	194
2	Ginny	1989-10-23	169
3	Demetra	1957-07-20	167
4	Sheri	1950-10-19	195
5	Karleen	2004-09-01	199
6	Daisey	2002-08-19	163
7	Audrey	2009-12-20	167
8	Alaine	1956-09-07	192
9	Edwin	1987-02-21	210
10	Shelli	1985-03-05	195
11	Santina	1991-09-04	164
12	Bart	1989-08-12	163
13	Harriette	1959-06-24	160
14	Jody	1962-12-18	202
15	Theodora	1995-11-08	204
16	Roman	1964-12-14	171
17	Jack	1976-06-11	167
18	Daphine	1998-09-21	191
19	Kyra	1966-05-04	202
\.

In [ ]:
TRUNCATE public.reliability;
COPY public.reliability (person, score) FROM stdin;
0	0.23828493492944236
1	0.657319818187148019
2	0.745325911826738019
3	0.656730287512349964
4	0.942979116189337052
5	0.600921893448834954
6	0.874435606539356036
7	0.990416985535926053
8	0.59251775051353095
9	0.688247502287665958
10	0.939401152561129993
11	0.960847979674174013
12	0.818769283596453956
13	0.834442059579594053
14	0.788371825897704048
15	0.620618845450902956
16	0.977769943596806024
17	0.840542782838639035
19	0.681836780693319988
\.

In [ ]:
TRUNCATE public.room;
COPY public.room (id, name, area) FROM stdin;
0	Dining room	23
1	Blue bedroom	20
2	Red bedroom	31
3	Yellow bedroom	27
4	Green bedroom	37
5	Living room	14
6	Kitchen	18
7	First bathroom	26
8	Second bathroom	34
9	Library	27
\.

In [ ]:
TRUNCATE public.sightings;
COPY public.sightings ("time", person, room, witness) FROM stdin;
02:30:00	19	8	0
05:00:00	11	9	0
03:00:00	19	2	0
13:00:00	8	8	0
22:30:00	5	1	0
05:30:00	19	5	0
16:00:00	11	8	0
18:30:00	11	9	0
13:30:00	11	4	0
13:30:00	6	7	0
05:30:00	13	3	0
10:00:00	5	0	0
04:00:00	14	6	0
01:30:00	12	1	0
15:00:00	1	5	0
21:00:00	16	6	0
06:30:00	17	6	1
01:30:00	10	5	1
09:30:00	9	7	1
07:00:00	17	1	1
10:30:00	3	3	1
01:00:00	18	2	1
09:00:00	17	2	1
05:30:00	18	6	1
04:30:00	16	2	2
15:30:00	14	8	2
19:00:00	1	8	2
22:00:00	5	9	2
22:30:00	0	0	2
18:00:00	10	3	2
06:00:00	11	5	2
05:00:00	17	8	2
17:00:00	14	3	2
17:30:00	12	9	2
22:30:00	10	3	2
21:00:00	5	7	2
09:00:00	9	4	2
08:30:00	18	2	2
10:00:00	13	3	2
23:00:00	7	9	2
13:30:00	5	6	3
19:00:00	16	3	3
03:00:00	16	4	3
12:30:00	16	0	3
20:30:00	8	0	3
14:00:00	15	1	3
22:00:00	8	3	3
10:00:00	15	7	3
11:00:00	15	3	3
00:00:00	15	4	3
22:00:00	14	9	3
02:30:00	15	7	4
08:00:00	11	6	4
15:00:00	13	3	4
20:00:00	8	7	4
21:00:00	7	3	4
19:00:00	15	7	4
22:30:00	9	6	5
06:00:00	0	1	5
02:30:00	0	5	5
17:30:00	1	1	5
18:00:00	7	4	5
04:30:00	18	3	5
14:30:00	17	9	5
21:30:00	15	4	5
10:00:00	1	9	5
03:00:00	3	0	5
05:30:00	3	8	5
19:30:00	17	4	6
16:30:00	0	5	7
11:00:00	1	9	7
13:30:00	18	8	7
13:00:00	12	9	7
19:30:00	3	9	7
20:30:00	3	0	7
15:00:00	6	3	7
19:30:00	6	7	7
19:30:00	10	5	7
13:00:00	5	3	8
15:00:00	14	2	8
01:00:00	1	6	8
08:00:00	7	4	8
09:30:00	12	1	8
20:30:00	12	9	8
10:30:00	11	9	8
06:30:00	7	0	8
11:30:00	13	1	8
15:30:00	5	0	9
04:00:00	6	1	9
22:30:00	2	5	9
01:00:00	8	6	9
13:30:00	15	1	9
07:00:00	19	9	9
21:00:00	7	2	9
18:00:00	0	3	9
08:30:00	14	0	9
03:30:00	11	9	9
05:30:00	3	6	9
20:00:00	15	3	9
06:00:00	7	6	9
16:00:00	14	1	9
19:00:00	7	1	10
12:00:00	17	8	10
09:00:00	8	7	10
21:00:00	8	1	10
01:00:00	8	0	10
09:30:00	17	5	10
08:00:00	3	4	10
21:00:00	18	1	10
22:00:00	3	4	10
11:00:00	15	2	10
01:30:00	18	1	10
08:00:00	14	0	10
06:00:00	7	2	10
04:00:00	18	2	10
21:00:00	12	4	10
01:00:00	4	0	10
18:30:00	13	4	10
22:00:00	5	1	10
23:30:00	11	6	10
04:30:00	5	3	11
04:30:00	12	2	11
13:30:00	7	1	11
08:30:00	7	9	11
00:00:00	6	7	11
11:30:00	6	1	11
20:00:00	6	2	11
07:00:00	9	6	11
10:00:00	16	2	11
04:00:00	8	0	11
07:30:00	15	1	11
20:30:00	10	9	11
19:30:00	3	0	11
04:30:00	4	7	11
12:00:00	1	0	11
23:30:00	17	9	11
18:00:00	4	4	11
21:00:00	0	6	11
14:30:00	17	8	12
07:30:00	10	8	12
05:30:00	13	7	12
19:30:00	18	5	12
21:30:00	8	4	12
21:30:00	11	5	12
13:30:00	3	9	12
08:00:00	2	2	12
08:00:00	5	1	12
01:00:00	13	7	12
15:00:00	19	3	12
21:30:00	3	3	12
11:00:00	12	2	13
04:30:00	3	0	13
02:30:00	3	9	13
05:30:00	5	5	13
01:00:00	1	5	13
09:00:00	15	2	13
22:00:00	18	7	13
18:30:00	7	7	13
08:30:00	18	7	13
09:30:00	6	1	13
21:00:00	6	5	13
16:30:00	19	5	13
15:30:00	1	6	14
07:30:00	7	9	14
04:30:00	13	2	14
10:00:00	17	9	14
07:30:00	12	5	14
15:30:00	8	6	14
10:00:00	18	9	14
18:00:00	0	6	14
17:30:00	2	7	14
18:30:00	5	5	14
04:00:00	4	8	14
12:30:00	7	4	14
00:30:00	19	5	14
14:30:00	9	1	14
09:00:00	3	9	14
14:00:00	7	2	14
00:30:00	12	6	14
16:00:00	8	4	15
23:00:00	12	1	15
13:30:00	18	2	15
11:30:00	2	4	15
00:00:00	10	9	15
00:30:00	3	7	15
03:30:00	3	1	15
00:30:00	0	2	15
16:30:00	10	2	15
08:00:00	8	6	15
06:00:00	2	2	15
03:00:00	13	1	15
06:00:00	8	5	15
15:00:00	18	3	15
01:30:00	3	0	15
02:30:00	5	8	15
22:30:00	19	7	15
22:00:00	15	4	16
10:30:00	0	5	17
17:00:00	1	5	17
12:30:00	5	3	17
00:00:00	19	7	17
12:00:00	1	7	17
16:00:00	5	7	17
14:00:00	3	8	17
14:30:00	14	0	17
04:00:00	6	3	19
11:00:00	4	5	19
15:30:00	5	5	19
\.

In [ ]:
ALTER TABLE ONLY public.person DROP CONSTRAINT IF EXISTS person_pkey;
ALTER TABLE ONLY public.person
    ADD CONSTRAINT person_pkey PRIMARY KEY (id);

In [ ]:
ALTER TABLE ONLY public.reliability DROP CONSTRAINT IF EXISTS reliability_pkey;
ALTER TABLE ONLY public.reliability
    ADD CONSTRAINT reliability_pkey PRIMARY KEY (person);

In [ ]:
ALTER TABLE ONLY public.room DROP CONSTRAINT IF EXISTS room_pkey;
ALTER TABLE ONLY public.room
    ADD CONSTRAINT room_pkey PRIMARY KEY (id);

In [ ]:
ALTER TABLE ONLY public.reliability DROP CONSTRAINT IF EXISTS reliability_person_fkey;
ALTER TABLE ONLY public.reliability
    ADD CONSTRAINT reliability_person_fkey FOREIGN KEY (person) REFERENCES public.person(id);

In [ ]:
ALTER TABLE ONLY public.sightings DROP CONSTRAINT IF EXISTS sightings_person_fkey;
ALTER TABLE ONLY public.sightings
    ADD CONSTRAINT sightings_person_fkey FOREIGN KEY (person) REFERENCES public.person(id);

In [ ]:
ALTER TABLE ONLY public.sightings DROP CONSTRAINT IF EXISTS sightings_room_fkey;
ALTER TABLE ONLY public.sightings
    ADD CONSTRAINT sightings_room_fkey FOREIGN KEY (room) REFERENCES public.room(id);

In [ ]:
ALTER TABLE ONLY public.sightings DROP CONSTRAINT IF EXISTS sightings_witness_fkey;
ALTER TABLE ONLY public.sightings
    ADD CONSTRAINT sightings_witness_fkey FOREIGN KEY (witness) REFERENCES public.person(id);

This creates four tables:

- `person` – the 20 persons present at the manor
- `room` – the rooms of the manor
- `sightings` – witness statements: who was seen where, and when
- `reliability` – the reliability score (between 0 and 1) of each witness

## Step 1: Explore the Database

Get familiar with the tables, for example through Studio's Schema panel.

## Step 2: Build a Sightings Table

Design a query that retrieves, for every sighting: the time, the name of the person seen, the name of the witness, and the name of the room. Store the result in a new table `s`.

In [ ]:
DROP TABLE IF EXISTS s CASCADE;
CREATE TABLE s AS
SELECT
  time,
  person.name  AS person,
  p2.name      AS witness,
  room.name    AS room
FROM sightings
  JOIN person       ON person  = person.id
  JOIN person AS p2 ON witness = p2.id
  JOIN room         ON room    = room.id;

## Step 3: Enable Provenance

Activate provenance tracking on the table `s` using [`add_provenance`](https://provsql.org/doxygen-sql/html/group__table__management.html#ga33ff696dabb05d813f2c1f914cb97d9a):

In [ ]:
SELECT add_provenance('s');

A hidden `provsql` column is added that holds a UUID provenance token for each tuple. Run a simple query to see this extra column in the output:

In [ ]:
SELECT * FROM s;

**Note:**

> The `provsql` column behaves specially: you cannot filter or sort on it directly. Use the [`provenance()`](https://provsql.org/doxygen-sql/html/group__provenance__output.html#gacb0ee8a16e9a316f163f4508a0e7c15d) function to obtain the current row’s token in expressions.

Create a *provenance mapping* that associates each provenance token with the name of the witness who made the sighting, using [`create_provenance_mapping`](https://provsql.org/doxygen-sql/html/group__table__management.html#gadbe591dd8145104c97ccd62e9dff5a7e):

In [ ]:
DROP TABLE IF EXISTS witness_mapping;
SELECT create_provenance_mapping('witness_mapping', 's', 'witness');

The mapping is stored as an ordinary table – inspect it with `SELECT * FROM witness_mapping;`.

## Step 4: Find Contradictions

Some witnesses are unreliable: the same person may be reported in two different rooms at the same time – an impossibility. Write a query that identifies all such contradictions.

In [ ]:
SELECT s1.time, s1.person, s1.room
FROM s AS s1, s AS s2
WHERE s1.person = s2.person
  AND s1.time   = s2.time
  AND s1.room  <> s2.room;

## Step 5: Display Provenance Formulas

Extend the previous query by adding `sr_formula(provenance(), 'witness_mapping')` to the `SELECT` clause to see *which witnesses* are responsible for each contradiction.

[`sr_formula`](https://provsql.org/doxygen-sql/html/group__compiled__semirings.html#ga3aad4775805b92307f3ca9e1b9fbd4c5) displays the provenance token as a formula, substituting each leaf token with the mapped value from `witness_mapping`.

In [ ]:
SELECT s1.time, s1.person, s1.room,
       sr_formula(provenance(), 'witness_mapping')
FROM s AS s1, s AS s2
WHERE s1.person = s2.person
  AND s1.time   = s2.time
  AND s1.room  <> s2.room;

## Step 6: Build a Consistent Sightings Table

Create a table `consistent_s` containing all sightings *except* those identified as contradictions. Display its content along with the provenance formula for each tuple.

In [ ]:
DROP TABLE IF EXISTS consistent_s CASCADE;
CREATE TABLE consistent_s AS
SELECT time, person, room FROM s
EXCEPT
SELECT s1.time, s1.person, s1.room
FROM s AS s1, s AS s2
WHERE s1.person = s2.person
  AND s1.time   = s2.time
  AND s1.room  <> s2.room;

SELECT *, sr_formula(provenance(), 'witness_mapping')
FROM consistent_s;

## Step 7: Identify Suspects

The murder happened between midnight and 8 am in a bedroom. Create a `suspects` table containing every person who was seen (in a consistent sighting) in a bedroom during that window. Display the suspects along with their provenance formula.

In [ ]:
DROP TABLE IF EXISTS suspects CASCADE;
CREATE TABLE suspects AS
SELECT DISTINCT person
FROM consistent_s
WHERE room LIKE '% bedroom'
  AND time BETWEEN '00:00:00' AND '08:00:00';

SELECT *, sr_formula(provenance(), 'witness_mapping')
FROM suspects;

## Step 8: Count Confirming Sightings

Use the counting m-semiring to find how many sightings confirm that each person is a suspect. Add an integer column `count` to `s`, set it to `1` for all rows, and create a `count_mapping`. Then use [`sr_counting`](https://provsql.org/doxygen-sql/html/group__compiled__semirings.html#gadc339f3d4c3289e3b6fa89a9331ec051) to display the count for each suspect.

In [ ]:
ALTER TABLE s ADD COLUMN IF NOT EXISTS count int;
UPDATE s SET count = 1;

DROP TABLE IF EXISTS count_mapping;
SELECT create_provenance_mapping('count_mapping', 's', 'count');

SELECT *, sr_counting(provenance(), 'count_mapping') AS c
FROM suspects
ORDER BY c;

## Step 9: Assign Reliability Probabilities

Add a `reliability` float column to `s` and populate it with the reliability score of each sighting’s witness. Then assign these scores as the probability of each provenance token using [`set_prob`](https://provsql.org/doxygen-sql/html/group__gate__manipulation.html#ga94dcc4874244a0c50c60608ab8506cca).

In [ ]:
ALTER TABLE s ADD COLUMN IF NOT EXISTS reliability float;

UPDATE s
SET reliability = score
FROM reliability, person
WHERE reliability.person = person.id
  AND person.name = s.witness;

SELECT set_prob(provenance(), reliability) FROM s;

## Step 10: Find the Murderer

The police needs a confidence of at least 0.99 before making an arrest. Use [`probability_evaluate`](https://provsql.org/doxygen-sql/html/group__probability.html#gad377e94cb1fff57141b1950cc4169c5e) to compute the probability that each suspect was truly present, and identify those above the threshold.

[`probability_evaluate`](https://provsql.org/doxygen-sql/html/group__probability.html#gad377e94cb1fff57141b1950cc4169c5e) accepts an optional second argument for the computation method:

- `'possible-worlds'` – exact, by exhaustive enumeration
- `'monte-carlo'` – approximate sampling (add a sample count as third argument)
- `'tree-decomposition'` – exact, via tree decomposition of the Boolean circuit
- `'compilation'` – d-DNNF compilation (add the tool name, e.g. `'d4'`, as third argument)

In [ ]:
SELECT *,
       sr_formula(provenance(), 'witness_mapping'),
       probability_evaluate(provenance(), 'possible-worlds')
FROM suspects
WHERE probability_evaluate(provenance(), 'possible-worlds') > 0.99
  AND person <> 'Daphine';